# **Mini-Project A2 Phase 1: Image Mask Manipulation with Semantic Segmentation**

In this project, you will learn to use segmentation and perform three major manipulatons to images, (1) Inpainting, (2) Styling, and (3) Replacing.

Please follow the notebook in order. Make sure to add code where indicated by `''' TODO '''` or `# YOUR CODE HERE`. Please answer all questions indicated with '❓', and answer where it says 'ANSWER HERE'.

Ensure your notebook is easy to follow. All written report answers should be provided in the notebook itself.

## **Overview**



1.   Segmentation
2.   Inpainting
3.   Styling
4.   Replacement


This lab will use 2 images, please add the paths accordingly! The TA will change and use their own images when grading.

Please use image1 and image2 throughout code so TAs can easily change images at top here!



In [ ]:
from PIL import Image
# Image1 and Image 2 paths, please use your own images!!

image1_path = "images/image1.jpg"
image2_path = "images/image2.jpg"

image1 = Image.open(image1_path).convert("RGB")
image2 = Image.open(image2_path).convert("RGB")
print("Loaded:", image1.size, image2.size)
image1


# **1. Segmentation**

We provide you with the code to run semantic segmentation using [DeepLabV3](https://pytorch.org/hub/pytorch_vision_deeplabv3_resnet101/) in pytorch below.

Note: You can run this without GPU, it will not be considerably slower since we are only doing inference on max 2 images this lab!!

We also provide the class names list below for masks. There are total 20 objects.

In [ ]:
from torchvision import transforms
from PIL import Image
import torch
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load model once (reuse across cells)
_weights = DeepLabV3_ResNet50_Weights.DEFAULT
seg_model = deeplabv3_resnet50(weights=_weights).to(device).eval()

def segmentation(image):
    """Run DeepLabV3 semantic segmentation. Returns HxW int mask of class indices."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = seg_model(image_tensor)['out'][0]
        segmentation_mask = output.argmax(0).detach().cpu().numpy()

    return segmentation_mask

class_names = [
    "Background",
    "Aeroplane",
    "Bicycle",
    "Bird",
    "Boat",
    "Bottle",
    "Bus",
    "Car",
    "Cat",
    "Chair",
    "Cow",
    "Dining Table",
    "Dog",
    "Horse",
    "Motorbike",
    "Person",
    "Potted Plant",
    "Sheep",
    "Sofa",
    "Train",
    "TV/Monitor"
]

def class_index(class_name):
    """Map class name (case-insensitive) to DeepLabV3 class index."""
    for i, name in enumerate(class_names):
        if name.lower() == class_name.lower():
            return i
    raise ValueError(f"Unknown class_name: {class_name}. Choose from {class_names}")


## **1.1 Visualization**

Like previous labs, you must visualize the outputs and images!!

Please complete `display_masks()` to show the original image, and all the masks detected, with their class names on top.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2


def display_masks(image, segmentation_mask=None, class_indices=None, class_names=class_names):
    """
    Show the original image and each detected class mask (excluding background).

    Accepts either:
      display_masks(image)                         # runs segmentation internally
      display_masks(image, mask, class_indices)    # uses a precomputed mask
    """
    if segmentation_mask is None:
        segmentation_mask = segmentation(image)

    present = np.unique(segmentation_mask)
    present = [int(c) for c in present if int(c) != 0]  # drop background
    if class_indices is not None:
        present = [int(c) for c in class_indices if int(c) != 0]

    n = 1 + len(present)
    cols = min(n, 4)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    for i, cls in enumerate(present, start=1):
        mask = (segmentation_mask == cls).astype(np.uint8) * 255
        axes[i].imshow(mask, cmap="gray")
        name = class_names[cls] if cls < len(class_names) else str(cls)
        axes[i].set_title(f"{name} (id={cls})")
        axes[i].axis("off")

    for j in range(1 + len(present), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    return segmentation_mask, present


In [ ]:
# Show image and all detected masks with class names (exclude background)
segmentation_mask = segmentation(image1)
class_indices = [int(c) for c in np.unique(segmentation_mask) if int(c) != 0]
display_masks(image1, segmentation_mask, class_indices)


In [ ]:
segmentation_mask2 = segmentation(image2)
class_indices2 = [int(c) for c in np.unique(segmentation_mask2) if int(c) != 0]
display_masks(image2, segmentation_mask2, class_indices2)


# **2. Inpainting: Object Removal**




Inpainting refers to the process of removing objects from an image by filling the corresponding regions (defined by a segmentation mask) with information, typically the surrounding background. This helps in maintaining visual continuity and gives the essence of object removal.

In this section, you will complete four functions, each implementing a different type of inpainting. Each method will replace the region defined by the segmentation mask, (given a class) with a unique technique.

Common Instructions:
- All functions should take in the image (PIL Image), and class name (as string) to inpaint.
- You can add additional parameters but set defaults!
- You can use OpenCV functions for various parts, the isntructions mention which ones are allowed. For any others, please confirm with TAs!

---

## **2.1 Mean Color Fill**

Complete `mean_color_fill()`.

**Description:** Replace the region defined by the segmentation mask with the mean RGB value of the surrounding region.

**Steps:**
1. Identify the pixels around the segmentation mask (the neighboring background).
2. Compute either: (a) Mean Color: Calculate the average RGB value of the surrounding pixels. (b) Majority Vote Color: Use the most frequent color in the surrounding pixels.
3. Replace all pixels in the mask with this computed value.



In [ ]:
def _to_np(image):
    return np.array(image.convert("RGB"))

def _get_binary_mask(image, class_name):
    mask = segmentation(image)
    idx = class_index(class_name)
    binary = (mask == idx).astype(np.uint8)
    if binary.sum() == 0:
        raise ValueError(f'Class "{class_name}" not found in image.')
    return binary

def mean_color_fill(image, class_name, ring=5, use_majority=False):
    """
    Replace the masked class region with the mean (or majority) RGB of a
    surrounding ring of background pixels.
    """
    img = _to_np(image).copy()
    binary = _get_binary_mask(image, class_name)

    # Neighboring background = dilated mask minus original mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * ring + 1, 2 * ring + 1))
    dilated = cv2.dilate(binary, kernel, iterations=1)
    surround = (dilated == 1) & (binary == 0)

    if surround.sum() == 0:
        # Fall back to whole-image mean excluding the object
        surround = binary == 0

    surround_pixels = img[surround]

    if use_majority:
        # Quantize slightly then take mode
        quantized = (surround_pixels // 8) * 8
        # majority via unique rows
        flat = quantized.astype(np.int32)
        # pack RGB into single int for bincount-like majority
        keys = (flat[:, 0] << 16) + (flat[:, 1] << 8) + flat[:, 2]
        vals, counts = np.unique(keys, return_counts=True)
        best = vals[counts.argmax()]
        fill = np.array([(best >> 16) & 255, (best >> 8) & 255, best & 255], dtype=np.uint8)
    else:
        fill = surround_pixels.mean(axis=0).astype(np.uint8)

    img[binary == 1] = fill
    return Image.fromarray(img)


## **2.2 Noise Fill**

Complete `noise_fill()`.

**Description:** Replace the masked region with a blurred and feathered version of the surrounding area to achieve smooth blending.

**Steps:**
1. Convert the segmentation mask into a binary mask for the selected class_index.
2. Feather the mask by applying dilation, expanding its edges to include some surrounding regions.
3. Create a blurred version of the image using a Gaussian blur with a large kernel size.
4. Blend the original image with the blurred version in the masked region, using the feathered mask as a weighting factor.

You may use `cv2.dilate()` for feathering and `cv2.GaussianBlur()` for blurring.

In [ ]:
def noise_fill(image, class_name, dilate_iters=8, blur_ksize=51):
    """
    Feather the mask with dilation, blur the whole image, then blend the
    blurred version into the masked region for a soft removal look.
    """
    img = _to_np(image).astype(np.float32)
    binary = _get_binary_mask(image, class_name)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    feathered = cv2.dilate(binary, kernel, iterations=dilate_iters).astype(np.float32)

    # Soft edge via blur on the mask itself
    k = blur_ksize if blur_ksize % 2 == 1 else blur_ksize + 1
    soft_mask = cv2.GaussianBlur(feathered, (k, k), 0)
    soft_mask = soft_mask / (soft_mask.max() + 1e-8)
    soft_mask = soft_mask[..., None]  # HxWx1

    blurred = cv2.GaussianBlur(img, (k, k), 0)
    blended = img * (1.0 - soft_mask) + blurred * soft_mask
    return Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8))


## **2.3 Linear Interpolation**

Complete `linear_interpolation()`. For this, you can use `cv2.inpaint()` opencv in-built function (see [documentation](https://docs.opencv.org/3.4/df/d3d/tutorial_py_inpainting.html)) or implement your own! Steps to implement are below.

**Description:** Blend the colors of the region edges to fill the gap with a linear interpolation.

**Steps:**
1. Identify the boundary of the segmentation mask and the pixel values along the edges.
2. Interpolate between these edge pixel values to estimate the colors for the missing region.
3. Fill the masked region with these interpolated values.

In [ ]:
def linear_interpolation(image, class_name, radius=3):
    """
    Fill the masked region using OpenCV Telea inpainting (edge-aware
    linear-style propagation of surrounding colors into the hole).
    """
    img = _to_np(image)
    binary = _get_binary_mask(image, class_name)
    # cv2.inpaint expects 8-bit single-channel mask, nonzeros = inpaint
    result = cv2.inpaint(img, binary, inpaintRadius=radius, flags=cv2.INPAINT_TELEA)
    return Image.fromarray(result)


## **2.4 Patch Inpaint**

Complete `patch_inpaint()`.

**Description:** Fill the masked region by replicating small patches from the surrounding area.

**Steps:**
1. Extract a small patch (e.g., 3x3 or 5x5) from the area around the mask. Patch size must be input to `patch_inpaint()`.
2. Start filling the mask from either:
(a) The top of the region, moving right, or
(b) The bottom, moving left.
3. After filling one row (or column) of the mask, extract a new patch from the surrounding area and repeat.

In [ ]:
def patch_inpaint(image, class_name, patch_size=5, direction="top_right"):
    """
    Fill the masked region by tiling small patches sampled from nearby
    background. patch_size must be provided (default 5).

    direction:
      "top_right"  — fill top-to-bottom, left-to-right
      "bottom_left"— fill bottom-to-top, right-to-left
    """
    if patch_size < 1:
        raise ValueError("patch_size must be >= 1")

    img = _to_np(image).copy()
    binary = _get_binary_mask(image, class_name).copy()
    h, w = binary.shape
    ys, xs = np.where(binary == 1)
    if len(ys) == 0:
        return Image.fromarray(img)

    y_min, y_max = int(ys.min()), int(ys.max())
    x_min, x_max = int(xs.min()), int(xs.max())
    mean_bg = img[binary == 0].mean(axis=0).astype(np.uint8)

    row_range = range(y_min, y_max + 1) if direction != "bottom_left" else range(y_max, y_min - 1, -1)

    def sample_patch(y, x):
        for radius in (patch_size, 2 * patch_size, 4 * patch_size, 8 * patch_size):
            for cy, cx in ((y - radius, x), (y + radius, x), (y, x - radius), (y, x + radius)):
                y0, x0 = cy, cx
                y1, x1 = cy + patch_size, cx + patch_size
                if y0 < 0 or x0 < 0 or y1 > h or x1 > w:
                    continue
                if binary[y0:y1, x0:x1].sum() == 0:
                    return img[y0:y1, x0:x1].copy()
        return None

    for y in row_range:
        patch = sample_patch(y, (x_min + x_max) // 2)
        if patch is None:
            patch = np.tile(mean_bg, (patch_size, patch_size, 1))
        cols = range(x_min, x_max + 1) if direction != "bottom_left" else range(x_max, x_min - 1, -1)
        for x in cols:
            if binary[y, x] == 0:
                continue
            y1 = min(h, y + patch_size)
            x1 = min(w, x + patch_size)
            ph, pw = y1 - y, x1 - x
            mask_block = binary[y:y1, x:x1] == 1
            img[y:y1, x:x1][mask_block] = patch[:ph, :pw][mask_block]
            binary[y:y1, x:x1][mask_block] = 0

    return Image.fromarray(img)


## **2.5 Visualize Inpainted images**

Please make a function, `visualize_inpaint()` to visualize all inpainted images side-by-side. The function should be able to take in an image, and class index to inpaint select class.

In [ ]:
def visualize_inpaint(image, class_name, patch_size=5):
    """Run all four inpaint methods and show them side-by-side."""
    methods = [
        ("Original", image),
        ("Mean Color Fill", mean_color_fill(image, class_name)),
        ("Noise Fill", noise_fill(image, class_name)),
        ("Linear Interpolation", linear_interpolation(image, class_name)),
        ("Patch Inpaint", patch_inpaint(image, class_name, patch_size=patch_size)),
    ]

    fig, axes = plt.subplots(1, len(methods), figsize=(4 * len(methods), 4))
    for ax, (title, im) in zip(axes, methods):
        ax.imshow(im)
        ax.set_title(title)
        ax.axis("off")
    plt.suptitle(f'Inpainting class: "{class_name}"')
    plt.tight_layout()
    plt.show()


In [ ]:
visualize_inpaint(image1, "Person")

In [ ]:
visualize_inpaint(image2, "Person")

### ❓ **Q 2.5 Explain the differences between all inpaint methods, and their advantages and disadvantages.**

**ANSWER:**

| Method | How it works | Advantages | Disadvantages |
|--------|--------------|------------|---------------|
| **Mean Color Fill** | Replaces the object with a single average (or majority) color from a ring of surrounding pixels. | Very fast and simple; good for small/flat backgrounds. | Looks flat and artificial; fails on textured or multi-colored backgrounds. |
| **Noise / Blur Fill** | Dilates the mask, blurs the image, and blends the blur into the hole with a soft feather. | Soft edges, fewer hard seams; quick. | Loses detail; large objects leave a smudgy blob. |
| **Linear Interpolation (`cv2.inpaint`)** | Propagates boundary colors into the hole (Telea / NS). | Often the most natural of the simple methods; respects local structure. | Can smear edges; struggles with large or highly textured holes. |
| **Patch Inpaint** | Copies small patches from nearby background into the mask row-by-row. | Can preserve local texture better than a flat fill. | Seams and tiling artifacts; slower; quality depends heavily on patch size and search. |

**Summary:** Mean fill is a baseline. Blur/noise is better for soft removal. OpenCV inpaint is usually the best automatic choice for moderate holes. Patch copy helps when you want texture but needs careful tuning.


# **3. Image Styling**

In this section, we will try to add styles to all unique segmentation masks!

Write a function called `apply_styles()` which takes an image as input and applies unique styles to each distinct **instance** in the mask. There can be multiple instances in a class!

You can specify styles like pixelation, invert, and more for the classes. You can use OpenCV to find styles you like. Make them look cool!

**Please ensure you have at least 4 unique styles and demonstrate them on your image by visualizing the output!**

*Hint*: you can use `cv2.connectedComponents()`.

In [ ]:
import numpy as np
import cv2
from PIL import Image
import random

def _style_pixelate(region, block=12):
    h, w = region.shape[:2]
    if h < block or w < block:
        return region
    small = cv2.resize(region, (max(1, w // block), max(1, h // block)), interpolation=cv2.INTER_LINEAR)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)

def _style_invert(region):
    return cv2.bitwise_not(region)

def _style_cartoon(region):
    # Edge-aware smoothing + edge overlay
    color = cv2.bilateralFilter(region, d=9, sigmaColor=75, sigmaSpace=75)
    gray = cv2.cvtColor(region, cv2.COLOR_RGB2GRAY)
    edges = cv2.adaptiveThreshold(cv2.medianBlur(gray, 7), 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                  cv2.THRESH_BINARY, 9, 2)
    edges = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    return cv2.bitwise_and(color, edges)

def _style_warm_tint(region):
    out = region.astype(np.float32)
    out[..., 0] = np.clip(out[..., 0] * 1.25, 0, 255)  # boost R
    out[..., 2] = np.clip(out[..., 2] * 0.75, 0, 255)  # reduce B
    return out.astype(np.uint8)

def _style_edge_enhance(region):
    gray = cv2.cvtColor(region, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 160)
    edges_rgb = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    return cv2.addWeighted(region, 0.7, edges_rgb, 0.3, 0)

def _style_posterize(region, levels=4):
    factor = 256 // levels
    return ((region // factor) * factor).astype(np.uint8)

DEFAULT_STYLES = [
    ("pixelate", _style_pixelate),
    ("invert", _style_invert),
    ("cartoon", _style_cartoon),
    ("warm_tint", _style_warm_tint),
    ("edge_enhance", _style_edge_enhance),
    ("posterize", _style_posterize),
]

def apply_styles(image, styles=None, show=True):
    """
    Apply a unique style to each connected-component *instance* of every
    non-background class in the segmentation mask. At least 4 styles are used.
    """
    if styles is None:
        styles = DEFAULT_STYLES

    img = _to_np(image).copy()
    mask = segmentation(image)
    present_classes = [int(c) for c in np.unique(mask) if int(c) != 0]

    style_idx = 0
    legend = []

    for cls in present_classes:
        binary = (mask == cls).astype(np.uint8)
        num_labels, labels = cv2.connectedComponents(binary)

        for inst in range(1, num_labels):  # 0 is background of this class
            name, fn = styles[style_idx % len(styles)]
            style_idx += 1

            inst_mask = labels == inst
            ys, xs = np.where(inst_mask)
            if len(ys) == 0:
                continue
            y0, y1 = ys.min(), ys.max() + 1
            x0, x1 = xs.min(), xs.max() + 1

            crop = img[y0:y1, x0:x1].copy()
            crop_mask = inst_mask[y0:y1, x0:x1]
            styled = fn(crop)
            crop[crop_mask] = styled[crop_mask]
            img[y0:y1, x0:x1] = crop

            cls_name = class_names[cls] if cls < len(class_names) else str(cls)
            legend.append(f"{cls_name}#{inst}: {name}")

    result = Image.fromarray(img)
    if show:
        plt.figure(figsize=(10, 6))
        plt.imshow(result)
        plt.title("Styled instances\n" + ", ".join(legend[:8]) + (" ..." if len(legend) > 8 else ""))
        plt.axis("off")
        plt.show()
        print("Styles applied:", legend)
    return result


In [ ]:
# Run on image 1
apply_styles(image1)

In [ ]:
# run on image 2
apply_styles(image2)

# **4. Replacement**

## **4.1 Replacement with semantic masks**

Write a function `replace_object()` which takes in two images, `image1` and `image2` and uses "copy paste" replacement.

**Goal:** Your goal is, given both images have at least one mask of same class, we will remove the class from `image1` (use one of the inpaint functions from Section 2) and replace it with a randomly chosen object of same class from `image2`.

- You must ensure both images have the class you want to replace.
- Your image2 masked must be resized appropriately to fit the inpainted region of image1!

- You may use OpenCV functions where necesary. Please confirm with TA the OpenCV functions you are using.

**Output:** Your output must be the modified image!

**Visualization:** Please show image1, image2 and replaced_image.

In [ ]:
def replace_object(image1, image2, class_name, inpaint_fn=linear_interpolation, return_intermediates=False):
    """
    Copy-paste replacement:
      1) Ensure both images contain `class_name`.
      2) Inpaint/remove that class from image1.
      3) Pick a random instance of the same class from image2.
      4) Resize it to fit image1's original object bounding box and paste.
    """
    img1 = _to_np(image1)
    img2 = _to_np(image2)

    mask1 = segmentation(image1)
    mask2 = segmentation(image2)
    idx = class_index(class_name)

    bin1 = (mask1 == idx).astype(np.uint8)
    bin2 = (mask2 == idx).astype(np.uint8)
    if bin1.sum() == 0:
        raise ValueError(f'"{class_name}" not found in image1')
    if bin2.sum() == 0:
        raise ValueError(f'"{class_name}" not found in image2')

    # Destination bbox from image1
    ys1, xs1 = np.where(bin1 == 1)
    y0, y1 = int(ys1.min()), int(ys1.max()) + 1
    x0, x1 = int(xs1.min()), int(xs1.max()) + 1
    dest_h, dest_w = y1 - y0, x1 - x0

    # Inpaint image1 to remove the object
    inpainted = np.array(inpaint_fn(image1, class_name).convert("RGB"))

    # Random instance from image2 via connected components
    num_labels, labels = cv2.connectedComponents(bin2)
    instances = list(range(1, num_labels))
    if not instances:
        raise ValueError(f'No connected instances of "{class_name}" in image2')
    chosen = random.choice(instances)
    inst_mask = (labels == chosen).astype(np.uint8)

    ys2, xs2 = np.where(inst_mask == 1)
    sy0, sy1 = int(ys2.min()), int(ys2.max()) + 1
    sx0, sx1 = int(xs2.min()), int(xs2.max()) + 1

    src_crop = img2[sy0:sy1, sx0:sx1]
    src_mask = inst_mask[sy0:sy1, sx0:sx1]

    resized_crop = cv2.resize(src_crop, (dest_w, dest_h), interpolation=cv2.INTER_LINEAR)
    resized_mask = cv2.resize(src_mask, (dest_w, dest_h), interpolation=cv2.INTER_NEAREST)

    replaced = inpainted.copy()
    region = replaced[y0:y1, x0:x1]
    region[resized_mask == 1] = resized_crop[resized_mask == 1]
    replaced[y0:y1, x0:x1] = region

    replaced_image = Image.fromarray(replaced)

    intermediates = {
        "image1": image1,
        "image2": image2,
        "inpainted_image1": Image.fromarray(inpainted),
        "selected_from_image2": Image.fromarray(
            np.where(src_mask[..., None] == 1, src_crop, 0).astype(np.uint8)
        ),
        "selected_mask": Image.fromarray((src_mask * 255).astype(np.uint8)),
        "replaced_image": replaced_image,
        "dest_bbox": (x0, y0, x1, y1),
        "chosen_instance": chosen,
    }

    if return_intermediates:
        return replaced_image, intermediates
    return replaced_image

# cache intermediates for show_intermediate()
_LAST_REPLACEMENT = {}


In [ ]:
replaced_image, _LAST_REPLACEMENT = replace_object(
    image1, image2, "Person", return_intermediates=True
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image1); axes[0].set_title("image1"); axes[0].axis("off")
axes[1].imshow(image2); axes[1].set_title("image2"); axes[1].axis("off")
axes[2].imshow(replaced_image); axes[2].set_title("replaced_image"); axes[2].axis("off")
plt.tight_layout()
plt.show()


## **4.2 More visualizations!**

Please make another function (`show_intermediate()`) to show all intermediate images from `replace_object()`. This includes, original `image1` and `image2`, inpainted `image1`, the selected masked region you chose from `image2`, and finally the output image.

In [ ]:
def show_intermediate(intermediates=None):
    """
    Show intermediate stages of replace_object:
    original image1/image2, inpainted image1, selected mask region from image2,
    and the final replaced output.
    """
    data = intermediates if intermediates is not None else _LAST_REPLACEMENT
    if not data:
        raise RuntimeError("No replacement intermediates cached. Run replace_object(..., return_intermediates=True) first.")

    panels = [
        ("image1", data["image1"]),
        ("image2", data["image2"]),
        ("inpainted image1", data["inpainted_image1"]),
        ("selected from image2", data["selected_from_image2"]),
        ("selected mask", data["selected_mask"]),
        ("replaced_image", data["replaced_image"]),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.reshape(-1)
    for ax, (title, im) in zip(axes, panels):
        ax.imshow(im, cmap="gray" if title == "selected mask" else None)
        ax.set_title(title)
        ax.axis("off")
    plt.suptitle("Replacement intermediates")
    plt.tight_layout()
    plt.show()

show_intermediate()


## ❓ **Q 4.3 Explain your replacement code.**

**ANSWER:**

`replace_object(image1, image2, class_name)` performs a copy-paste object swap using semantic masks:

1. **Segment both images** with DeepLabV3 and locate the requested class (e.g. Person).
2. **Measure the destination box** — the bounding box of that class in `image1`.
3. **Inpaint `image1`** with one of the Section 2 methods (default: `linear_interpolation` / `cv2.inpaint`) so the original object is removed cleanly.
4. **Pick a source instance from `image2`** using `cv2.connectedComponents` on the class mask, then choose one instance at random.
5. **Crop** that instance (RGB + mask), **resize** both to the destination box size, and **paste** only where the resized source mask is 1.

The result is `image1` with its object removed and replaced by a same-class object from `image2`, scaled to fit. `show_intermediate()` visualizes every stage of this pipeline.


# End of Phase 1 😊🥳